In [2]:
#importing data from hugging face 
from datasets import load_dataset

data = load_dataset("Allanatrix/Materials")
df = data["train"].to_pandas()

df.head(4)

c:\Users\kisho\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,material_id,formula_pretty,n_elements,contains_transition_metal,formation_energy_per_atom,energy_per_atom,band_gap,is_semiconductor,density,volume,elements
0,mp-1232999,LiO8,2,False,1.720359,0.654797,-0.765226,False,-1.181862,-0.911997,"['Li', 'O']"
1,mp-1236127,LiO8,2,False,1.719137,0.654475,-0.808019,False,-1.237528,-0.897719,"['Li', 'O']"
2,mp-1546006,LiS4,2,False,1.902530,0.036938,0.626478,True,-2.352166,4.289055,"['Li', 'S']"
3,mp-995393,LiS4,2,False,1.499043,0.862733,0.620655,True,-2.367226,4.875701,"['Li', 'S']"


Feature Engineering 

In [3]:

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
from pymatgen.core.periodic_table import Element
import numpy as np


In [4]:
def avg_atomic_number(formula):
    comp = Composition(formula)
    total_atoms = comp.num_atoms
    
    return sum(
        el.Z * amt / total_atoms
        for el, amt in comp.items()
    )
df['avg_atomic_number'] = df['formula_pretty'].apply(avg_atomic_number)

In [5]:
def avg_atomic_mass(formula):
    comp = Composition(formula)
    total_atoms = comp.num_atoms
    
    return sum(
        el.atomic_mass * amt / total_atoms
        for el, amt in comp.items()
    )
df['avg_atomic_mass'] = df['formula_pretty'].apply(avg_atomic_mass)

In [6]:
def electronegativity_mean(formula):
    comp = Composition(formula)
    total_atoms = comp.num_atoms
    
    values = [
        el.X * amt / total_atoms
        for el, amt in comp.items()
        if el.X is not None
    ]
    return sum(values)
df['electronegativity_mean'] = df['formula_pretty'].apply(electronegativity_mean)

In [7]:
def electronegativity_std(formula):
    comp = Composition(formula)
    
    xs = []
    weights = []
    
    for el, amt in comp.items():
        if el.X is not None:
            xs.append(el.X)
            weights.append(amt)
    
    xs = np.array(xs)
    weights = np.array(weights)
    
    mean = np.average(xs, weights=weights)
    variance = np.average((xs - mean)**2, weights=weights)
    
    return np.sqrt(variance)
df['electronegativity_std'] = df['formula_pretty'].apply(electronegativity_std)

In [8]:
def element_fractions(formula):
    comp = Composition(formula)
    total_atoms = comp.num_atoms
    
    return {
        str(el): amt / total_atoms
        for el, amt in comp.items()
    }
df['element_fractions'] = df['formula_pretty'].apply(element_fractions)

In [9]:
df.head(3)

,material_id,formula_pretty,n_elements,contains_transition_metal,formation_energy_per_atom,energy_per_atom,band_gap,is_semiconductor,density,volume,elements,avg_atomic_number,avg_atomic_mass,electronegativity_mean,electronegativity_std,element_fractions
0,mp-1232999,LiO8,2,False,1.720359,0.654797,-0.765226,False,-1.181862,-0.911997,"['Li', 'O']",7.444444,14.992911,3.166667,0.773103,"{'Li': 0.1111111111111111, 'O': 0.888888888888..."
1,mp-1236127,LiO8,2,False,1.719137,0.654475,-0.808019,False,-1.237528,-0.897719,"['Li', 'O']",7.444444,14.992911,3.166667,0.773103,"{'Li': 0.1111111111111111, 'O': 0.888888888888..."
2,mp-1546006,LiS4,2,False,1.902530,0.036938,0.626478,True,-2.352166,4.289055,"['Li', 'S']",13.400000,27.040200,2.260000,0.640000,"{'Li': 0.2, 'S': 0.8}"


In [10]:
df.drop(columns=['elements'],inplace=True)

In [11]:
import pandas as pd

element_df = pd.json_normalize(df['element_fractions'])
element_df = element_df.fillna(0)

df = pd.concat([df.drop(columns=['element_fractions']), element_df], axis=1)


In [12]:
y_formation_energy_per_atom = df['formation_energy_per_atom']
y_energy_per_atom = df['energy_per_atom']
y_band_gap = df['band_gap']
y_is_semiconductor = df['is_semiconductor']
y_density = df['density']
y_volume = df['volume']

In [13]:
df.drop(columns=['formation_energy_per_atom','energy_per_atom','band_gap','is_semiconductor','density','volume'],inplace=True)

In [14]:
df.head(4)

,material_id,formula_pretty,n_elements,contains_transition_metal,avg_atomic_number,avg_atomic_mass,electronegativity_mean,electronegativity_std,Li,O,...,Pu,Np,Re,Os,Ru,Ta,Sc,Tc,Th,Xe
0,mp-1232999,LiO8,2,False,7.444444,14.992911,3.166667,0.773103,0.111111,0.888889,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,mp-1236127,LiO8,2,False,7.444444,14.992911,3.166667,0.773103,0.111111,0.888889,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,mp-1546006,LiS4,2,False,13.400000,27.040200,2.260000,0.640000,0.200000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,mp-995393,LiS4,2,False,13.400000,27.040200,2.260000,0.640000,0.200000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
X = df.iloc[:,2:]

In [18]:
import joblib

joblib.dump(
    X.columns.tolist(),
    "C:/Materials and it's Mechanical Properties/Data/feature_columns.pkl"
)



["C:/Materials and it's Mechanical Properties/Data/feature_columns.pkl"]

In [139]:
import joblib 

joblib.dump(X, "C:\Materials and it's Mechanical Properties\Data\X_features.pkl")
joblib.dump(y_formation_energy_per_atom, "C:\Materials and it's Mechanical Properties\Data\y_formation_energy_per_atom.pkl")
joblib.dump(y_energy_per_atom, "C:\Materials and it's Mechanical Properties\Data\y_energy_per_atom.pkl")
joblib.dump(y_is_semiconductor, "C:\Materials and it's Mechanical Properties\Data\y_is_semiconductor.pkl")
joblib.dump(y_density, "C:\Materials and it's Mechanical Properties\Data\y_density.pkl")
joblib.dump(y_volume, "C:\Materials and it's Mechanical Properties\Data\y_volume.pkl")

<>:3: SyntaxWarning: invalid escape sequence '\M'
<>:4: SyntaxWarning: invalid escape sequence '\M'
<>:5: SyntaxWarning: invalid escape sequence '\M'
<>:6: SyntaxWarning: invalid escape sequence '\M'
<>:7: SyntaxWarning: invalid escape sequence '\M'
<>:8: SyntaxWarning: invalid escape sequence '\M'
<>:3: SyntaxWarning: invalid escape sequence '\M'
<>:4: SyntaxWarning: invalid escape sequence '\M'
<>:5: SyntaxWarning: invalid escape sequence '\M'
<>:6: SyntaxWarning: invalid escape sequence '\M'
<>:7: SyntaxWarning: invalid escape sequence '\M'
<>:8: SyntaxWarning: invalid escape sequence '\M'
C:\Users\kisho\AppData\Local\Temp\ipykernel_4632\3802727573.py:3: SyntaxWarning: invalid escape sequence '\M'
  joblib.dump(X, "C:\Materials and it's Mechanical Properties\Data\X_features.pkl")
C:\Users\kisho\AppData\Local\Temp\ipykernel_4632\3802727573.py:4: SyntaxWarning: invalid escape sequence '\M'
  joblib.dump(y_formation_energy_per_atom, "C:\Materials and it's Mechanical Properties\Data\y_f

["C:\\Materials and it's Mechanical Properties\\Data\\y_volume.pkl"]